In [2]:
%cd ..

/workspaces/legis_event_extractor


/workspaces/legis_event_extractor/.venv/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import spacy
import json
import numpy as np

In [4]:
def toSpacy(dataSet):
    spacy_data = []
    for entry in dataSet:
        text = str(entry["text"])
        entities = [(start, end, label) for start, end, label in entry["label"]]
        spacy_data.append((text, {"entities": entities}))
    return spacy_data

In [5]:
def SpacytoConLL(dataset):
    """
    Convierte un archivo JSON con texto y etiquetas a formato CoNLL usando spaCy.
    Args:
        input_file (str): Ruta del archivo JSON de entrada.

        model (str): Modelo de spaCy para tokenización.
    """
    # Cargar el modelo de spaCy
    nlp = spacy.load("es_core_news_lg") # python -m spacy download es_core_news_lg

    conLLData = []
    # Abrir el archivo de salida
    for entry in dataset:
        text = entry[0]
        labels = entry[1]["entities"]
        connTexto = ""
        # Procesar el texto con spaCy
        doc = nlp(text)

        # Inicializar etiquetas BIO
        tags = ["O"] * len(doc)

        # Asignar etiquetas según las entidades
        for start, end, label in labels:
            for token in doc:
                if token.idx >= start and token.idx < end:
                    if token.idx == start:
                        tags[token.i] = f"B-{label}"  # Inicio de la entidad
                    else:
                        tags[token.i] = f"I-{label}"  # Dentro de la entidad

            # Escribir cada token y su etiqueta en el archivo de salida
        for token, tag in zip(doc, tags):
            if token.text.strip() == "":
                connTexto += "\n"
            else:
                connTexto += f"{token.text} {tag}\n"

        conLLData.append(connTexto)
    return conLLData

In [6]:
file_path = "data/dataset.jsonl"

DataSet = []

# Leer todas las líneas como una lista y recorrerlas
with open(file_path, "r") as file:
    lines = file.readlines()
    for line in lines:
        data = json.loads(line)
        DataSet.append(data)

spacyDataSet = toSpacy(DataSet)

In [7]:
# para arreglar las etiquetas que inician/terminan en espacios en blanco
i = 0
for text, props in spacyDataSet:
    for index, row in enumerate(props["entities"]):
        start = row[0]
        end = row[1]
        while text[start] == " ":
            start += 1
            i += 1

        while text[end - 1] == " ":
            end -= 1
            i += 1

        my_list = list(row)

        # Modificar un elemento
        my_list[0] = start
        my_list[1] = end

        # Volver a convertir a tupla si es necesario
        props["entities"][index] = tuple(my_list)
print(i)

42


In [8]:
# Copyright 2025 codespace
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [9]:
import spacy
from spacy.training import offsets_to_biluo_tags


def align_offsets_to_tokens(doc, entities):
    """
    Alinea los offsets de las entidades a los límites de los tokens generados por spaCy.
    """
    aligned_entities = []
    for start, end, label in entities:
        token_start = None
        token_end = None
        for token in doc:
            # Encontrar el token que contiene el inicio de la entidad
            if token.idx <= start < token.idx + len(token.text):
                token_start = token.idx
            # Encontrar el token que contiene el final de la entidad
            if token.idx < end <= token.idx + len(token.text):
                token_end = token.idx + len(token.text)
        # Si ambos límites están definidos, añadir la entidad ajustada
        if token_start is not None and token_end is not None:
            aligned_entities.append((token_start, token_end, label))
    return aligned_entities

In [10]:
nlp = spacy.blank("es")


for text, props in spacyDataSet:
    doc = nlp.make_doc(text)
    props["entities"] = align_offsets_to_tokens(doc, props["entities"])

In [11]:
# Crear Daset ConLL

conLLDataset = SpacytoConLL(spacyDataSet)

In [12]:
def load_custom_dataset(data_list):
    """
    Convierte una lista de strings en un DatasetDict compatible con Hugging Face.
    Args:
        data_list (list): Lista de strings en formato CoNLL.
    Returns:
        DatasetDict: Conjunto de datos dividido en entrenamiento y prueba.
    """
    sentences = []
    ner_tags = []
    current_sentence = []
    current_tags = []

    for doc in data_list:
        for line in doc.split("\n"):
            if line.strip() == "":  # Nueva oración
                if current_sentence:
                    sentences.append(current_sentence)
                    ner_tags.append(current_tags)
                    current_sentence = []
                    current_tags = []
            else:
                token, tag = line.strip().split()
                current_sentence.append(token)
                current_tags.append(tag)

        # Añadir la última oración
        if current_sentence:
            sentences.append(current_sentence)
            ner_tags.append(current_tags)

    # Convertir etiquetas BIO a índices numéricos
    unique_tags = sorted(set(tag for tags in ner_tags for tag in tags))
    tag2id = {tag: i for i, tag in enumerate(unique_tags)}
    id2tag = {i: tag for tag, i in tag2id.items()}

    # Transformar etiquetas en índices
    ner_tags = [[tag2id[tag] for tag in tags] for tags in ner_tags]

    # Crear un DatasetDict
    dataset = DatasetDict(
        {
            "train": Dataset.from_dict(
                {"tokens": sentences[: int(0.8 * len(sentences))], "ner_tags": ner_tags[: int(0.8 * len(sentences))]}
            ),
            "test": Dataset.from_dict(
                {"tokens": sentences[int(0.8 * len(sentences)) :], "ner_tags": ner_tags[int(0.8 * len(sentences)) :]}
            ),
        }
    )

    return dataset, unique_tags, id2tag, tag2id

In [13]:
from datasets import load_dataset, DatasetDict, Dataset


dataset, labels, id2label, label2id = load_custom_dataset(conLLDataset)

print(id2label)
print(label2id)
print(labels)

/workspaces/legis_event_extractor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{0: 'B-AUTOR', 1: 'B-DESTINO', 2: 'B-EVENTO', 3: 'B-MATERIA', 4: 'I-AUTOR', 5: 'I-DESTINO', 6: 'I-EVENTO', 7: 'I-MATERIA', 8: 'O'}
{'B-AUTOR': 0, 'B-DESTINO': 1, 'B-EVENTO': 2, 'B-MATERIA': 3, 'I-AUTOR': 4, 'I-DESTINO': 5, 'I-EVENTO': 6, 'I-MATERIA': 7, 'O': 8}
['B-AUTOR', 'B-DESTINO', 'B-EVENTO', 'B-MATERIA', 'I-AUTOR', 'I-DESTINO', 'I-EVENTO', 'I-MATERIA', 'O']


In [14]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

# 1. Configuración inicial
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForTokenClassification(tokenizer)

In [15]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer


# 4. Tokenización
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label[word_id])
            else:
                label_ids.append(-100)
            previous_word_id = word_id
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [16]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|██████████| 118/118 [00:00<00:00, 2903.82 examples/s]


In [17]:
# 5. Cargar modelo
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of BertForTokenClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
# 6. Configurar argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./data/ner_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
)

In [26]:
# 7. Métrica de evaluación
from sklearn.metrics import classification_report


def compute_metrics(pred):
    labels = pred.label_ids.flatten()
    preds = np.argmax(pred.predictions, axis=2).flatten()
    true_labels = [id2label[label] for label in labels if label != -100]
    true_preds = [id2label[pred] for pred, label in zip(preds, labels) if label != -100]
    report = classification_report(true_labels, true_preds, output_dict=True, zero_division=0)
    return {
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
    }

In [28]:
# 8. Entrenador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

/tmp/ipykernel_113791/2351471764.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 9. Entrenamiento
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.598400,0.444724,0.803961,0.868621,0.815738
